# Human Validation Analysis
This notebook reproduces the human validation results reported in 
Section 5 and Table 3 of the paper:
> **Beyond Black-Box Labels: Interpretable Criteria for Diagnosing Subjective NLP Tasks**

It tests whether diagnostic co-activation signals align with where 
domain experts actually disagree when assigning single labels.

**Key design choice:** All 6 LLM annotators (including Gemini) are used 
for the diagnostic activation, matching the original paper computation.
The main stability/overlap analysis (notebooks 01, 02) uses 5 models 
(excluding Gemini) as reported in Section 4.2.

**Reproduces Table 3:**
| Pair | Human split (%) | Diag. co-act (%) |
|------|----------------|-----------------|
| c1–c2 | 37.5 | 83.8 |
| c1–c3 | 16.5 | 44.4 |
| c2–c3 | 14.9 | 50.7 |

## 1. Setup

In [86]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import cohen_kappa_score

# ── Paths ─────────────────────────────────────────────────────────
TENSOR_NPY  = Path("../data/tensors/run_2026-04-13_14-00_FULL_AGGREGATED/tensor_raw_4700.npy")
MAPPING_CSV = Path("../data/tensors/run_2026-04-13_14-00_FULL_AGGREGATED/mapping_sentences.csv")
HUMAN_CSV   = Path("../data/anonymized/human_annotations_merged.csv")
ANNOT_DIR   = Path("../data/original/human_annotation")

# ── Constants ─────────────────────────────────────────────────────
EXPERTS = ["label_E1","label_E2","label_E3","label_E4","label_E5"]

LABEL_MAP = {
    "Retour sur investissement": "c1",
    "Notoriété":                 "c2",
    "Obligation":                "c3",
    "Description":               "c0",
}

# Category mapping
# c1: Performance & Efficiency   → q01,q02,q03 (indices 0,1,2)
# c2: User Experience & Brand    → q04,q05,q06 (indices 3,4,5)
# c3: Obligation & Safety        → q07,q08,q09 (indices 6,7,8)
CATEGORY_MAP = {
    "c1": [0, 1, 2],
    "c2": [3, 4, 5],
    "c3": [6, 7, 8],
}

N_CRITERIA = 9
t = 1  # engagement threshold

print("Setup complete.")

Setup complete.


## 2. Load human annotations

In [87]:
# ── Load ALL 500 rows (including repeats for reliability) ─────────
human_raw = pd.read_csv(HUMAN_CSV)
human_raw['sentence_id'] = human_raw['sentence_id'].astype(int)
print(f"Raw annotations  : {human_raw.shape}")
print(f"Unique sentences : {human_raw['sentence_id'].nunique()}")

# ── Majority vote per sentence ────────────────────────────────────
def majority_vote(s):
    m = s.dropna().mode()
    return m.iloc[0] if len(m) > 0 else np.nan

human_agg = human_raw.groupby('sentence_id').agg(
    client_uid=('client_uid', 'first'),
    doc_uid=('doc_uid', 'first'),
    **{exp: (exp, majority_vote) for exp in EXPERTS}
).reset_index()

print(f"After majority vote: {len(human_agg)} unique sentences")

Raw annotations  : (389, 8)
Unique sentences : 389
After majority vote: 389 unique sentences


## 3. Load tensor and compute diagnostic activation (all 6 models)

In [88]:
# ── Load tensor ───────────────────────────────────────────────────
tensor  = np.load(TENSOR_NPY)
mapping = pd.read_csv(MAPPING_CSV)

# All 6 models for human validation (matches paper)
tensor_all6 = tensor[:, :N_CRITERIA, :]  # (4700, 9, 6)
print(f"Tensor shape (6 models): {tensor_all6.shape}")

# ── Get tensor rows for human sentences ───────────────────────────
human_ids     = set(human_raw['sentence_id'].unique())
human_mapping = mapping[mapping['sentence_id'].isin(human_ids)].copy()
print(f"Human sentences in mapping: {len(human_mapping)}")

# ── Extract and sum votes ─────────────────────────────────────────
h_idx        = human_mapping['tensor_idx'].values
tensor_votes = tensor_all6[h_idx]
tensor_votes = np.where(tensor_votes == -1, 0, tensor_votes)
vote_counts  = tensor_votes.sum(axis=2)  # (N, 9)

# ── Focus sets and category activation ───────────────────────────
focus = vote_counts >= t
c1    = focus[:, CATEGORY_MAP['c1']].any(axis=1)
c2    = focus[:, CATEGORY_MAP['c2']].any(axis=1)
c3    = focus[:, CATEGORY_MAP['c3']].any(axis=1)
m_st  = c1.astype(int) + c2.astype(int) + c3.astype(int)

print(f"Covered (m_st>=1): {(m_st>=1).sum()}")
print(f"c1 active: {c1.sum()} | c2 active: {c2.sum()} | c3 active: {c3.sum()}")

# ── Diagnostic dataframe ──────────────────────────────────────────
diag_df = pd.DataFrame({
    'sentence_id': human_mapping['sentence_id'].astype(int).values,
    'c1_active'  : c1.astype(int),
    'c2_active'  : c2.astype(int),
    'c3_active'  : c3.astype(int),
    'm_st'       : m_st,
})
print(f"diag_df: {diag_df.shape}")

Tensor shape (6 models): (4700, 9, 6)
Human sentences in mapping: 389
Covered (m_st>=1): 365
c1 active: 316 | c2 active: 347 | c3 active: 203
diag_df: (389, 5)


## 4. Table 3: Boundary alignment

In [89]:
# ── Merge 1: majority vote for diag co-act ────────────────────────
merged_agg  = human_agg.merge(diag_df, on='sentence_id', how='inner')
covered_agg = merged_agg[merged_agg['m_st'] >= 1].copy()

# ── Merge 2: all 500 rows for human split ────────────────────────
merged_all  = human_raw.merge(diag_df, on='sentence_id', how='inner')
covered_all = merged_all[merged_all['m_st'] >= 1].copy()

print(f"Covered (majority vote) : {len(covered_agg)}")
print(f"Covered (all 500 rows)  : {len(covered_all)}")

# ── Table 3 ───────────────────────────────────────────────────────
print(f"\nTable 3: Boundary alignment")
print(f"{'Pair':<8} {'Human split':>20} {'Diag co-act':>20}")

for ca, cb in [("c1","c2"),("c1","c3"),("c2","c3")]:
    # Human split: all 500 rows, covered denominator
    has_a = (covered_all[EXPERTS] == ca).any(axis=1)
    has_b = (covered_all[EXPERTS] == cb).any(axis=1)
    hn = (has_a & has_b).sum()
    ht = len(covered_all)

    # Diag co-act: majority vote, covered denominator
    dn = ((covered_agg[f'{ca}_active']==1) &
          (covered_agg[f'{cb}_active']==1)).sum()
    n  = len(covered_agg)

    print(f"{ca}-{cb}   {hn}/{ht}={hn/ht*100:.1f}%   {dn}/{n}={dn/n*100:.1f}%")

Covered (majority vote) : 365
Covered (all 500 rows)  : 365

Table 3: Boundary alignment
Pair              Human split          Diag co-act
c1-c2   145/365=39.7%   306/365=83.8%
c1-c3   61/365=16.7%   162/365=44.4%
c2-c3   59/365=16.2%   185/365=50.7%


**Note on reproducibility:** Human split rates may differ slightly 
from paper values (~2%) due to deduplication of repeated annotation 
items. Diagnostic co-activation rates (83.8%, 44.4%, 50.7%) 
reproduce exactly.
